# ARC Atlanta — the one-click notebook

Reproduce the **Atlanta Regional Commission AM highway assignment** and validate it
against ARC's own count benchmark (**expected: region-wide %RMSE ≈ 22 %**, ARC target ~38 %).

**`Run All` is safe.** By default it runs the fast preflight, prepares the run folder, and
does a **1-iteration smoke run (~2 min)**. The **full 6,031-zone equilibrium (~5–6 min) is
never launched without you flipping `RUN_MODE = "full"`** in the cell below — a real-scale
run must never start by accident.

Every cell calls one stage of [`arc_pipeline.py`](arc_pipeline.py) — the same pipeline you
can drive stage-by-stage from a terminal (`python arc_pipeline.py check | convert |
prepare | run | validate | all`). One source of truth, three doors (CLI, this notebook,
[`START_HERE.md`](../../START_HERE.md)).

| Stage | What | Time |
|---|---|---|
| check | deps, kernel, data audit, intake gate, VDF/PLF verify | seconds |
| convert | PATH B only (full raw ARC data → `gmns/`); **skips with a clear message otherwise** | – |
| prepare | copy `gmns/` verbatim + set solver params explicitly | seconds |
| run `--quick` | 1-iteration smoke, live streamed output | ~2 min |
| run (full) | equilibrium, converges ~iteration 8 | ~5–6 min |
| validate | %RMSE by volume group vs ARC counts | seconds |

**Two facts that confuse everyone, settled up front:**
1. The full raw ARC data (~125 MB shapefiles + trip cores) is **NOT in this repo — that is
   expected, not an error.** The bundled `gmns/` case is complete (PATH A).
2. `gmns/link.csv` **already encodes ARC's calibration** (per-FACTYPE modified-BPR,
   weave overrides, `vdf_plf = 3.66/4 = 0.915`). The pipeline **verifies and copies** it —
   nothing rewrites your network files.

Works on Windows, macOS, Linux — the kernel path is resolved per platform
(macOS: `brew install cmake libomp`, then `bash build.sh` at the repo root; without
libomp it still builds, just serial/slower).

In [ ]:
import arc_pipeline as ap

RUN_MODE = "quick"   # "quick" = 1-iteration smoke (~2 min)  |  "full" = real run (~5-6 min)
                     # | "none" = preflight + prepare only, no kernel launch
QUICK = RUN_MODE == "quick"
print(f"RUN_MODE = {RUN_MODE!r} -- the full-scale run only happens if YOU set it to 'full'")

## 1 — Preflight check *(seconds, never launches the kernel)*
Dependencies → kernel binary (platform-aware) → data audit (PATH A/B) → **intake gate**
(`GATE: READY` means every modeling convention — capacity basis, period, PLF, VDF, demand
units, allowed-use, tolls — is *declared* in `gmns/submission.yml`, not guessed) →
**VDF/PLF verification** (confirms `gmns/link.csv` still matches ARC Sec 7.1.2; if it has
drifted you get a FAIL here instead of a silently wrong assignment).

In [ ]:
assert ap.stage_check(), "fix the first FAIL above, then re-run this cell"

## 2 — Convert *(PATH B only — skips cleanly on PATH A)*
Only does work if you placed the full raw ARC data in `arc-Shape/arc-Shape/` and
`TODAM20_asgn/`; then it rebuilds `gmns/` from scratch (`arc_atlanta_to_gmns.py` +
`arc_demand_to_csv.py`). Otherwise it prints SKIP — **that is the normal case.**

In [ ]:
assert ap.stage_convert()

## 3 — Prepare the run folder *(verify + copy, set solver explicitly)*
Verifies the encoded calibration again, copies `gmns/` **verbatim** into `gmns_run/`
(or `gmns_run_quick/`), then writes the solver parameters — iterations, relative-gap stop,
processors, AM 6–10 window — **explicitly, printed line by line.** This is the only place
solver settings come from. One demand period, strictly serial stages, one kernel process.

In [ ]:
if RUN_MODE != "none":
    assert ap.stage_prepare(quick=QUICK)
else:
    assert ap.stage_prepare()
    print("\nPrepared only. To run later from a terminal:\n"
          "  python arc_pipeline.py run --quick   # ~2 min smoke\n"
          "  python arc_pipeline.py run           # full ~5-6 min\n"
          "  python arc_pipeline.py validate")

## 4 — Run the kernel *(live output — quiet start is normal)*
⏳ The first ~minute shows little output: the kernel is reading ~26 M OD pairs and building
the first shortest-path trees. The elapsed clock and 30-second heartbeat below prove it is
alive. After that, one equilibrium iteration ≈ 30 s; the full run converges around
iteration 8 (relative gap < 0.5 % three times in a row).

Full console output is also saved to `gmns_run*/kernel_console.log`.

In [ ]:
if RUN_MODE == "none":
    print("RUN_MODE='none' -- skipping the kernel by request")
else:
    assert ap.stage_run(quick=QUICK), "see kernel_console.log in the run folder"

## 5 — Validate against ARC's count benchmark
Joins assigned `volume` to `arc_am_ref_volume.csv` (ARC's own AM auto volumes,
`V_SOVAM+V_HOV2AM+V_HOV3AM`) by (from, to) and scores **%RMSE by volume group** against
ARC Sec 7.1.4 thresholds.

- **Full run:** every group passes, region-wide %RMSE ≈ 22 % (target ~38 %), assigned/ref ≈ 1.00.
- **Quick run:** the number is approximate (1 iteration ≠ equilibrium) and labeled INFO.

If a full run looks wrong, suspect a **convention mismatch** (units, PLF, VDF, demand
class), not the solver — that is what stages 1 and 3 verify first.

In [ ]:
if RUN_MODE == "none":
    print("RUN_MODE='none' -- nothing to validate yet")
else:
    ap.stage_validate(quick=QUICK)
ap.summary()

## Where to go next
- **Ran the smoke test?** Set `RUN_MODE = "full"` in the first code cell and re-run cells
  3–5 to get the real, converged 22 % validation.
- **Why each convention matters:** [README.md](README.md) §4 (PLF pitfall, modified BPR),
  [ARC_BENCHMARK.md](ARC_BENCHMARK.md), [ARC_DTALite_kernel_requirements.md](ARC_DTALite_kernel_requirements.md)
- **2× faster scenario runs:** [SUPERZONE.md](SUPERZONE.md) (`python arc_superzone.py 1500`),
  then recover the original 6,031×6,031 skim with `python arc_skim.py sz / compare`
- **Onboard YOUR agency the same way:** `docs/MPO_ONBOARDING_GUIDE.md` +
  `docs/GOLDEN_PATH_CHECKLIST.md`
- **Smaller first steps:** `kernel/data_sets/03_chicago_sketch`,
  `notebooks/00_install_and_quickstart.ipynb`